<a href="https://colab.research.google.com/github/poonam-021/explainable-ai-stress-burnout/blob/main/models/Visual_Analysis_Of_Stress.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CORE TRAINING PIPELINE

In [1]:
import os           # For navigating folders and file paths
import random       # Python's built-in random number generator

# --- Data & Math ---
import numpy as np          # Arrays, matrix math, pixel manipulation
import pandas as pd         # DataFrames — used heavily for CK+ CSV

# --- Image processing ---
import cv2                  # OpenCV: reading, resizing, colour conversion

# --- Visualisation ---
import matplotlib.pyplot as plt   # Plotting images, charts, curves
import matplotlib.patches as mpatches  # For custom legend patches
import seaborn as sns              # Beautiful statistical plots

# --- Progress bars (so long loops don't feel like they froze) ---
from tqdm import tqdm

# --- Path handling (cleaner than raw strings) ---
from pathlib import Path

# --- TensorFlow / Keras (our deep learning framework) ---
import tensorflow as tf
from tensorflow import keras

# --- Collections (for counting) ---
from collections import Counter

# ── Fix ALL random seeds so results are 100% reproducible ───
SEED = 42
random.seed(SEED)                          # Python random
np.random.seed(SEED)                       # NumPy random
tf.random.set_seed(SEED)                   # TensorFlow random
os.environ['PYTHONHASHSEED'] = str(SEED)   # Hash-based randomness in Python

# ── Print versions so we know our environment ───────────────
print("=" * 55)
print("  ✅  All imports loaded successfully!")
print("=" * 55)
print(f"  NumPy     : {np.__version__}")
print(f"  Pandas    : {pd.__version__}")
print(f"  OpenCV    : {cv2.__version__}")
print(f"  TensorFlow: {tf.__version__}")
print(f"  GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("=" * 55)
import os           # For navigating folders and file paths
import random       # Python's built-in random number generator

# --- Data & Math ---
import numpy as np          # Arrays, matrix math, pixel manipulation
import pandas as pd         # DataFrames — used heavily for CK+ CSV

# --- Image processing ---
import cv2                  # OpenCV: reading, resizing, colour conversion

# --- Visualisation ---
import matplotlib.pyplot as plt   # Plotting images, charts, curves
import matplotlib.patches as mpatches  # For custom legend patches
import seaborn as sns              # Beautiful statistical plots

# --- Progress bars (so long loops don't feel like they froze) ---
from tqdm import tqdm

# --- Path handling (cleaner than raw strings) ---
from pathlib import Path

# --- TensorFlow / Keras (our deep learning framework) ---
import tensorflow as tf
from tensorflow import keras

# --- Collections (for counting) ---
from collections import Counter

# ── Fix ALL random seeds so results are 100% reproducible ───
SEED = 42
random.seed(SEED)                          # Python random
np.random.seed(SEED)                       # NumPy random
tf.random.set_seed(SEED)                   # TensorFlow random
os.environ['PYTHONHASHSEED'] = str(SEED)   # Hash-based randomness in Python

print("=" * 55)
print("  ✅  All imports loaded successfully!")
print("=" * 55)
print(f"  NumPy     : {np.__version__}")
print(f"  Pandas    : {pd.__version__}")
print(f"  OpenCV    : {cv2.__version__}")
print(f"  TensorFlow: {tf.__version__}")
print(f"  GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("=" * 55)


  ✅  All imports loaded successfully!
  NumPy     : 2.0.2
  Pandas    : 2.2.2
  OpenCV    : 4.13.0
  TensorFlow: 2.19.0
  GPU available: False
  ✅  All imports loaded successfully!
  NumPy     : 2.0.2
  Pandas    : 2.2.2
  OpenCV    : 4.13.0
  TensorFlow: 2.19.0
  GPU available: False


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Copy zip from Drive to Colab's fast local disk
!cp /content/drive/MyDrive/raf-db.zip /content/

# Unzip quietly (-q) into a dedicated folder (-d)
!unzip -q /content/raf-db.zip -d /content/raf_data/

print("✅  RAF-DB extracted to /content/raf_data/")

✅  RAF-DB extracted to /content/raf_data/


In [4]:
!cp /content/drive/MyDrive/ck+.zip /content/
!unzip -q /content/ck+.zip -d /content/cf_data/

print("✅  CK+ extracted to /content/cf_data/")

✅  CK+ extracted to /content/cf_data/


In [5]:
#RAF-DB paths
RAF_TRAIN_DIR = Path('/content/raf_data/raf-db/train')  # 7 emotion folders
RAF_TEST_DIR  = Path('/content/raf_data/raf-db/test')   # same structure


In [6]:
#CK+ path
CK_CSV_PATH = Path('/content/cf_data/ck+/ckextended.csv')

In [7]:
# ── Target image size (EfficientNetV2-S expects 224×224×3) ───
IMG_SIZE = 224      # Both height and width
IMG_CHANNELS = 3    # RGB (3 colour channels)

In [8]:
STRESS    = 1       # Label for stressed images
NO_STRESS = 0       # Label for non-stressed images

In [9]:
#RAF-DB emotion label mapping
# Folder number (as a string) → emotion name → binary label
# We store the name too so our EDA prints are readable
RAF_EMOTION_NAMES = {
    '1': 'Surprise',   # → No-Stress
    '2': 'Fear',       # → Stress
    '3': 'Disgust',    # → Stress
    '4': 'Happy',      # → No-Stress
    '5': 'Sad',        # → Stress
    '6': 'Anger',      # → Stress
    '7': 'Neutral',    # → No-Stress
}

# Set of folder numbers that map to STRESS=1
RAF_STRESS_FOLDERS = {'2', '3', '5', '6'}   # Fear, Disgust, Sad, Anger


In [10]:
#CK+ emotion label mapping
# CK+ uses DIFFERENT numbers than RAF-DB
CK_EMOTION_NAMES = {
    0: 'Anger',      # → Stress
    1: 'Contempt',   # → Discard (ambiguous, not used)
    2: 'Disgust',    # → Stress
    3: 'Fear',       # → Stress
    4: 'Happy',      # → No-Stress
    5: 'Sadness',    # → Stress
    6: 'Surprise',   # → No-Stress
    7: 'Neutral',    # → No-Stress
}

# CK+ emotion integers that are stress-related
CK_STRESS_EMOTIONS = {0, 2, 3, 5}    # Anger, Disgust, Fear, Sadness


In [11]:
#Training constants
BATCH_SIZE    = 32     # Images per training step
EPOCHS_STAGE1 = 10     # Frozen backbone training
EPOCHS_STAGE2 = 20     # Fine-tuning training
LEARNING_RATE = 1e-3   # Initial learning rate

In [12]:
paths_to_check = {
    "RAF Train Dir" : RAF_TRAIN_DIR,
    "RAF Test Dir"  : RAF_TEST_DIR,
    "CK+ CSV"       : CK_CSV_PATH,
}

In [13]:
all_ok = True
for name, path in paths_to_check.items():
    exists = path.exists()
    status = "✅" if exists else "❌  NOT FOUND"
    print(f"  {status}  {name}: {path}")
    if not exists:
        all_ok = False

print("=" * 55)
if all_ok:
    print("  ✅  All paths verified. Ready to proceed!")
else:
    print("  ❌  Fix the missing paths above before continuing.")

  ✅  RAF Train Dir: /content/raf_data/raf-db/train
  ✅  RAF Test Dir: /content/raf_data/raf-db/test
  ✅  CK+ CSV: /content/cf_data/ck+/ckextended.csv
  ✅  All paths verified. Ready to proceed!
